# Técnicas de Diseño y Validación de Modelos · Universidad Blas Pascal
## Módulo 4 · Técnicas de Regularización
## Laboratorio: Ridge, Lasso y ElasticNet con selección de λ por CV

<div style="display:flex; justify-content:flex-start; margin: 8px 0 18px 0;">
  <a href="https://colab.research.google.com/github/GabrielaOjcius/Laboratorios-Tecnicas-Diseno-Validacion-Modelos/blob/main/Laboratorios%20opcionales/M4_Notebook.ipynb" target="_blank">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/>
  </a>
  <span style="font-size: 0.98em;">En Colab, hac&eacute; una copia en tu Drive antes de editar.</span>
</div>


---

### Objetivo
Implementar las tres penalizaciones clásicas sobre datos donde se conoce la solución verdadera. Esto permite evaluar no solo el R² en test, sino también la calidad de la **recuperación de la estructura subyacente**: ¿selecciona Lasso las variables correctas? ¿Las ordena correctamente en el camino de regularización?

### Alineación con el material teórico
Corresponde a la **Actividad Optativa** de `Módulo 4 · Técnicas de Regularización`. Prepara la **Sección B** de la Parte 2 y la **Sección 4** de la Evaluación Integradora.

### Instrucciones
1. Ejecutar en orden (Entorno de ejecución → Ejecutar todo).
2. Completar las celdas marcadas con `# ✏️ RESPONDA AQUÍ`.
3. No modificar el código de setup (B.1) sin justificación.

In [ ]:
# ─── B.1. Datos con estructura conocida ──────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, ElasticNet,
    RidgeCV, LassoCV, ElasticNetCV, lasso_path
)
from sklearn.pipeline import Pipeline

# Dataset con 50 predictores, 8 informativos — guardamos los coeficientes reales
X, y, coef_real = make_regression(
    n_samples=150, n_features=50, n_informative=8,
    noise=15.0, coef=True, random_state=7
)
idx_info = np.where(coef_real != 0)[0]  # índices de los 8 predictores informativos
print(f'Features informativas (índices): {idx_info}')
print(f'Coeficientes reales no nulos: {len(idx_info)} de {coef_real.size}')

# Partición 75 % train / 25 % test
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=7)

# Estandarización ajustada SOLO sobre train (sin leakage)
scaler = StandardScaler().fit(X_tr)
X_tr_s = scaler.transform(X_tr)
X_te_s = scaler.transform(X_te)

print(f'Train: {X_tr_s.shape}  |  Test: {X_te_s.shape}')

In [ ]:
# ─── B.2. Ridge: efecto de lambda sobre los coeficientes ─────────────────────
print(f"{'Alpha (λ)':>10}  {'||β||²':>10}  {'R² test':>8}  {'Coef nulos':>10}")
for a in [0.0, 0.1, 1.0, 10.0, 100.0, 1000.0]:
    if a == 0.0:
        m = LinearRegression().fit(X_tr_s, y_tr)
    else:
        m = Ridge(alpha=a).fit(X_tr_s, y_tr)
    norma = np.sum(m.coef_**2)
    nnz = int(np.sum(np.abs(m.coef_) < 1e-6))
    r2te = m.score(X_te_s, y_te)
    print(f"{a:>10.1f}  {norma:>10.2f}  {r2te:>8.3f}  {nnz:>10d}")

In [ ]:
# ─── B.3. Lasso: selección automática de variables ───────────────────────────
print(f"{'Alpha (λ)':>10}  {'No nulos':>10}  {'R² test':>8}")
for a in [0.01, 0.1, 0.5, 1.0, 5.0, 20.0]:
    m = Lasso(alpha=a, max_iter=10000).fit(X_tr_s, y_tr)
    nnz = int(np.sum(np.abs(m.coef_) > 1e-6))
    r2te = m.score(X_te_s, y_te)
    print(f"{a:>10.2f}  {nnz:>10d}  {r2te:>8.3f}")

In [ ]:
# ─── B.4. Selección automática de λ por validación cruzada ───────────────────
alphas = np.logspace(-3, 3, 80)
cv = KFold(5, shuffle=True, random_state=42)

ols        = LinearRegression().fit(X_tr_s, y_tr)
ridge_cv   = RidgeCV(alphas=alphas, cv=cv).fit(X_tr_s, y_tr)
lasso_cv   = LassoCV(alphas=alphas, cv=cv, max_iter=10000, random_state=42).fit(X_tr_s, y_tr)
enet_cv    = ElasticNetCV(alphas=alphas, l1_ratio=[0.2, 0.5, 0.8],
                          cv=cv, max_iter=10000, random_state=42).fit(X_tr_s, y_tr)

print(f"{'Modelo':14}  {'alpha':>8}  {'R² train':>9}  {'R² test':>8}  {'||β||₂':>8}  {'No nulos':>9}")
for nombre, modelo in [('OLS', ols), ('Ridge CV', ridge_cv),
                        ('Lasso CV', lasso_cv), ('ElasticNet CV', enet_cv)]:
    alpha_val = getattr(modelo, 'alpha_', float('nan'))
    nnz = int(np.sum(np.abs(modelo.coef_) > 1e-6))
    norma = np.linalg.norm(modelo.coef_)
    r2tr = modelo.score(X_tr_s, y_tr)
    r2te = modelo.score(X_te_s, y_te)
    print(f"{nombre:14}  {alpha_val:>8.4f}  {r2tr:>9.3f}  {r2te:>8.3f}  {norma:>8.2f}  {nnz:>9d}")

In [ ]:
# ─── B.5. Camino de regularización del Lasso ─────────────────────────────────
alphas_path = np.logspace(-2, 2.5, 80)
_, coefs, _ = lasso_path(X_tr_s, y_tr, alphas=alphas_path)

plt.figure(figsize=(10, 5.5))
for i in range(coefs.shape[0]):
    es_info = i in idx_info
    plt.plot(alphas_path, coefs[i],
             color='#8B1A2F' if es_info else 'lightgray',
             lw=2.2 if es_info else 0.8,
             alpha=0.95 if es_info else 0.5)

plt.axvline(lasso_cv.alpha_, color='black', ls='--', lw=1.5,
            label=f'λ óptimo (LassoCV) = {lasso_cv.alpha_:.3f}')
plt.xscale('log')
plt.xlabel('λ (escala log)'); plt.ylabel('Coeficiente')
plt.title('Camino de regularización del Lasso\n(rojo: features informativas | gris: no informativas)')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ─── B.6. ¿Lasso recupera las features informativas? ─────────────────────────
idx_sel = np.where(np.abs(lasso_cv.coef_) > 1e-6)[0]
tp = set(idx_sel) & set(idx_info)
fp = set(idx_sel) - set(idx_info)
fn = set(idx_info) - set(idx_sel)

print(f'Features informativas reales:     {len(idx_info)}')
print(f'Features seleccionadas por Lasso: {len(idx_sel)}')
print(f'  Verdaderos positivos (correctas): {len(tp)} / {len(idx_info)}')
print(f'  Falsos positivos (espurias incl.): {len(fp)}')
print(f'  Falsos negativos (informativas perdidas): {len(fn)}')

# Comparación visual coeficientes reales vs. estimados
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Coeficientes reales vs. estimados', fontweight='bold')
for ax, modelo, nombre in zip(axes,
                               [ridge_cv, lasso_cv, enet_cv],
                               ['Ridge CV', 'Lasso CV', 'ElasticNet CV']):
    ax.scatter(coef_real, modelo.coef_, alpha=0.6, color='#8B1A2F')
    lim = max(abs(coef_real).max(), abs(modelo.coef_).max()) * 1.1
    ax.plot([-lim, lim], [-lim, lim], '--', color='gray', lw=1)
    ax.set_xlabel('Coeficiente real'); ax.set_ylabel('Coeficiente estimado')
    ax.set_title(nombre); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ─── B.7. Exploración: ¿qué pasa con 40/50 features informativas? ────────────
X2, y2, _ = make_regression(
    n_samples=150, n_features=50, n_informative=40,
    noise=15.0, coef=True, random_state=7
)
Xtr2, Xte2, ytr2, yte2 = train_test_split(X2, y2, test_size=0.25, random_state=7)
sc2 = StandardScaler().fit(Xtr2)
Xtr2_s, Xte2_s = sc2.transform(Xtr2), sc2.transform(Xte2)

r2_cv  = RidgeCV(alphas=alphas, cv=cv).fit(Xtr2_s, ytr2)
l2_cv  = LassoCV(alphas=alphas, cv=cv, max_iter=10000, random_state=42).fit(Xtr2_s, ytr2)

print('=== Comparación entre escenarios ===')
print(f'               8 /50 informativas      40/50 informativas')
print(f'Ridge R² test: {ridge_cv.score(X_te_s, y_te):.3f}                  {r2_cv.score(Xte2_s, yte2):.3f}')
print(f'Lasso R² test: {lasso_cv.score(X_te_s, y_te):.3f}                  {l2_cv.score(Xte2_s, yte2):.3f}')

---
## Preguntas de análisis
Responder en las siguientes celdas. Referenciar valores numéricos de las tablas de B.4 y los gráficos de B.5 y B.6.

### Pregunta 1
¿Cuántos coeficientes anula el Lasso con su λ óptimo? ¿Coinciden con los predictores no informativos? Reportar el número de TP, FP y FN de la selección.

**Respuesta:**

`# ✏️ RESPONDA AQUÍ`

### Pregunta 2
¿Qué R² en test obtiene cada modelo? ¿Cuál generaliza mejor en el escenario original (8/50 informativos)? ¿Por qué OLS tiene el peor R² pese a usar todas las variables?

**Respuesta:**

`# ✏️ RESPONDA AQUÍ`

### Pregunta 3
Observando el camino de regularización (B.5): ¿entran primero al modelo las features informativas (rojas) o las no informativas (grises) cuando λ disminuye desde el máximo? ¿Qué interpretación tiene ese orden?

**Respuesta:**

`# ✏️ RESPONDA AQUÍ`

### Pregunta 4
Los λ óptimos de Ridge y Lasso, ¿son comparables directamente entre sí? Justificar con la forma matemática de cada penalización.

**Respuesta:**

`# ✏️ RESPONDA AQUÍ`

### Pregunta 5
Con n_informative=40 (la mayoría de los 50 features son relevantes), ¿qué método funciona mejor? ¿Por qué Ridge se vuelve más competitivo en ese escenario? Conectar con la propiedad de sparsity de Lasso.

**Respuesta:**

`# ✏️ RESPONDA AQUÍ`